In [1]:
# ================================================================
# CELL 1 — Config & imports
# ================================================================
import json, time, requests
import networkx as nx
import pandas as pd
from pyvis.network import Network
from IPython.display import IFrame

WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"
HEADERS = {"User-Agent": "ELgoldSubgraph/1.0 (student project)"}

CAT_COLORS = {
    "News":                   "#E85D24",
    "Job_offers":             "#3B8BD4",
    "Movie_reviews":          "#1D9E75",
    "Automotive_blogs":       "#BA7517",
    "Amazon_product_reviews": "#7F77DD",
    "Scientific_abstracts":   "#E91E8C",
    "Historic_blogs":         "#888780",
}

PROP_LABELS = {
    "P31": "est un",       "P279": "sous-classe de",
    "P17": "pays",         "P131": "situé dans",
    "P495": "pays d'origine", "P57": "réalisateur",
    "P161": "acteur",      "P19":  "lieu naissance",
    "P27":  "nationalité", "P106": "profession",
    "P361": "partie de",   "P127": "propriétaire",
}

with open("elgold_entities_per_domain.json") as f:
    elgold = json.load(f)

qids_by_category = {cat: set(data["entities_qids"]) for cat, data in elgold.items()}
all_qids = set().union(*qids_by_category.values())
# mapping qid -> titre Wikipedia (pour lookup rapide)
title_by_qid = {}
for cat, data in elgold.items():
    for title, qid in data["title_to_qid"].items():
        title_by_qid[qid] = title

print(f"Entités uniques : {len(all_qids)} QIDs sur {len(elgold)} catégories")

Entités uniques : 1447 QIDs sur 7 catégories


In [5]:
# ================================================================
# CELL 2 — Lecture depuis wikidata_elgold.json
# ================================================================
import json

with open("wikidata_elgold.json", encoding="utf-8") as f:
    dump = json.load(f)

label_map, type_map, triplets = {}, {}, []
PROPS_WANTED = set(PROP_LABELS.keys())

for qid, entity in dump.items():
    if qid not in all_qids:
        continue

    # --- Label ---
    labels = entity.get("labels", {})
    label_map[qid] = labels.get("en", {}).get("value", qid)

    # --- Type (P31) ---
    claims = entity.get("claims", {})
    if "P31" in claims:
        try:
            t = claims["P31"][0]["mainsnak"]["datavalue"]["value"]["id"]
            if t in dump:
                type_map[qid] = dump[t].get("labels", {}).get("en", {}).get("value", t)
            else:
                type_map[qid] = t
        except (KeyError, IndexError):
            pass

    # --- Relations inter-entités ---
    for prop in PROPS_WANTED:
        if prop not in claims:
            continue
        for snak in claims[prop]:
            try:
                obj_qid = snak["mainsnak"]["datavalue"]["value"]["id"]
                if obj_qid in all_qids:
                    triplets.append({"s": qid, "p": prop, "o": obj_qid})
            except (KeyError, TypeError):
                continue

print(f"✓ {len(label_map)} labels")
print(f"✓ {len(triplets)} relations inter-entités ELgold")

✓ 1447 labels
✓ 1990 relations inter-entités ELgold


In [7]:
# ================================================================
# CELL 3 — Construction des sous-graphes par thème + visualisation
# ================================================================

qid_to_cats = {}
for cat, qids in qids_by_category.items():
    for q in qids:
        qid_to_cats.setdefault(q, []).append(cat)

# --- Graphe global (base pour extraire les sous-graphes) ---
G = nx.DiGraph()

for qid, cats in qid_to_cats.items():
    label = label_map.get(qid, title_by_qid.get(qid, qid))
    G.add_node(qid,
               label=label,
               title=f"<b>{label}</b><br>{'<br>'.join(cats)}<br><i>{type_map.get(qid,'')}</i>",
               color=CAT_COLORS[cats[0]],
               size=6 + 4 * len(cats),
               group=cats[0])

for t in triplets:
    if G.has_node(t["s"]) and G.has_node(t["o"]):
        prop_label = PROP_LABELS.get(t["p"], t["p"])
        G.add_edge(t["s"], t["o"], label=prop_label, title=prop_label)

# --- Extraction et visualisation par thème ---
for cat, qids in qids_by_category.items():
    nodes_in_cat = [q for q in qids if G.has_node(q)]
    subG = G.subgraph(nodes_in_cat).copy()

    print(f"\n{cat} : {subG.number_of_nodes()} nœuds | {subG.number_of_edges()} arêtes")

    # Visualisation
    net = Network(height="800px", width="100%", directed=True,
                  bgcolor="#1a1a2e", font_color="white")
    net.from_nx(subG)
    net.set_options("""{
      "physics": {
        "forceAtlas2Based": {
          "gravitationalConstant": -80,
          "centralGravity": 0.003,
          "springLength": 120,
          "springConstant": 0.06,
          "damping": 0.4
        },
        "solver": "forceAtlas2Based",
        "stabilization": { "iterations": 200 }
      },
      "edges": {
        "smooth": { "type": "continuous" },
        "arrows": { "to": { "enabled": true, "scaleFactor": 0.4 } }
      },
      "interaction": { "hover": true, "navigationButtons": true }
    }""")
    net.save_graph(f"graph_{cat}.html")

    # Export CSV
    pd.DataFrame([
        {"qid": n,
         "label": subG.nodes[n]["label"],
         "type": type_map.get(n, ""),
         "categories": cat}
        for n in subG.nodes
    ]).to_csv(f"nodes_{cat}.csv", index=False)

    triplets_cat = [t for t in triplets if t["s"] in nodes_in_cat and t["o"] in nodes_in_cat]
    pd.DataFrame(triplets_cat).to_csv(f"edges_{cat}.csv", index=False)

    print(f"  ✓ graph_{cat}.html")
    print(f"  ✓ nodes_{cat}.csv")
    print(f"  ✓ edges_{cat}.csv")


News : 342 nœuds | 426 arêtes
  ✓ graph_News.html
  ✓ nodes_News.csv
  ✓ edges_News.csv

Job_offers : 188 nœuds | 178 arêtes
  ✓ graph_Job_offers.html
  ✓ nodes_Job_offers.csv
  ✓ edges_Job_offers.csv

Movie_reviews : 243 nœuds | 287 arêtes
  ✓ graph_Movie_reviews.html
  ✓ nodes_Movie_reviews.csv
  ✓ edges_Movie_reviews.csv

Automotive_blogs : 160 nœuds | 116 arêtes
  ✓ graph_Automotive_blogs.html
  ✓ nodes_Automotive_blogs.csv
  ✓ edges_Automotive_blogs.csv

Amazon_product_reviews : 245 nœuds | 152 arêtes
  ✓ graph_Amazon_product_reviews.html
  ✓ nodes_Amazon_product_reviews.csv
  ✓ edges_Amazon_product_reviews.csv

Scientific_abstracts : 287 nœuds | 103 arêtes
  ✓ graph_Scientific_abstracts.html
  ✓ nodes_Scientific_abstracts.csv
  ✓ edges_Scientific_abstracts.csv

Historic_blogs : 102 nœuds | 87 arêtes
  ✓ graph_Historic_blogs.html
  ✓ nodes_Historic_blogs.csv
  ✓ edges_Historic_blogs.csv


In [ ]:
subgraphs = {}
for cat, qids in qids_by_category.items():
    # noeuds présents dans G appartenant à cette catégorie
    nodes_in_cat = [q for q in qids if G.has_node(q)]
    subgraphs[cat] = G.subgraph(nodes_in_cat).copy()
    print(f"{cat}: {subgraphs[cat].number_of_nodes()} nœuds, "
          f"{subgraphs[cat].number_of_edges()} arêtes")

News: 342 nœuds, 311 arêtes
Job_offers: 188 nœuds, 124 arêtes
Movie_reviews: 243 nœuds, 220 arêtes
Automotive_blogs: 160 nœuds, 86 arêtes
Amazon_product_reviews: 245 nœuds, 114 arêtes
Scientific_abstracts: 287 nœuds, 67 arêtes
Historic_blogs: 102 nœuds, 57 arêtes
